# Diabetic Retinopathy Classification

Multi-class CNN classifier for diabetic retinopathy stages (0-4) using the preprocessed `data_preprocessed` dataset.

**Preprocessing pipeline** (run once via `preprocess_images.py`):
- Crop retinal disk (remove black background via Gaussian blur + binary mask + bounding box)
- Resize to 512×512 (LANCZOS)
- Saves to `data_preprocessed/` with same train/val/test/class structure

**Best configuration** (found after 4 rounds of systematic experiments, 24+ configurations):
- Architecture: EfficientNetB0 (pretrained ImageNet)
- Optimizer: AdamW, lr=3e-4, weight_decay=1e-4
- Scheduler: OneCycleLR
- Loss: CrossEntropyLoss + inverse-frequency class weights + label_smoothing=0.1
- **No WeightedRandomSampler** (simpler config outperforms sampler with full dataset)

**Results on full dataset** (4864 total images, 3652 train / 727 val / 485 test):
- Best single model test accuracy: **0.5938** (EffB0+Simple head, no sampler)
- 2-model ensemble test accuracy: **0.5649**

Classes: 0-noDR | 1-mild | 2-moderate | 3-severe | 4-proliferativeDR


In [1]:
import os
os.chdir('/Users/edu/Documents/Universidad/Master_Ingeniería_Informática/clases/iaiao/p3')
print(f'Working directory: {os.getcwd()}')

Working directory: /Users/edu/Documents/Universidad/Master_Ingeniería_Informática/clases/iaiao/p3


In [2]:
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import EfficientNet_B0_Weights

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import random, os, time
import numpy as np
from PIL import Image
from collections import Counter

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Using device: {device}')

data_split_dir = 'data_preprocessed'
model_dir = 'models'
os.makedirs(model_dir, exist_ok=True)

CLASS_NAMES = ['0-noDR', '1-mild', '2-moderate', '3-severe', '4-proliferativeDR']
NUM_CLASSES = len(CLASS_NAMES)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')


Using device: mps
Classes (5): ['0-noDR', '1-mild', '2-moderate', '3-severe', '4-proliferativeDR']


## Preprocessing: Retinal Disk Crop

Fundus photographs have large black borders (27–54% of pixels) outside the circular retinal disk.
Cropping these borders before training reduces noise and lets the model focus on relevant anatomy.

**Algorithm** (implemented in `preprocess_images.py`, run offline):
1. Gaussian blur (radius=5) to suppress JPEG artifacts at the border
2. Binary mask: pixels with max-channel value > 15 are retina
3. Axis-aligned bounding box of the mask + 3% margin
4. Square-pad the shorter axis, then crop
5. Resize to 512×512 (LANCZOS)

Result: black background dropped from ~52% → ~28% (remaining black is disk-circle corners).


In [3]:
from PIL import ImageFilter

def crop_retinal_disk(img: Image.Image, threshold: int = 15, margin: float = 0.03) -> Image.Image:
    """
    Crop a fundus image to a tight square around the retinal disk.
    Removes the large black background borders typical in fundus photography.

    Args:
        img:       PIL RGB image.
        threshold: Max-channel value below which a pixel is background (0-255).
        margin:    Extra fractional border to keep around the detected disk.

    Returns:
        Square-cropped PIL image, or the original if no foreground is found.
    """
    arr_blur = np.array(img.filter(ImageFilter.GaussianBlur(radius=5)))
    mask = arr_blur.max(axis=2) > threshold
    h, w = arr_blur.shape[:2]
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    if not rows.any() or not cols.any():
        return img  # entirely black – return unchanged
    rmin = int(np.where(rows)[0][0])
    rmax = int(np.where(rows)[0][-1])
    cmin = int(np.where(cols)[0][0])
    cmax = int(np.where(cols)[0][-1])
    rh = rmax - rmin
    cw = cmax - cmin
    pad = int(max(rh, cw) * margin)
    rmin = max(0, rmin - pad)
    rmax = min(h - 1, rmax + pad)
    cmin = max(0, cmin - pad)
    cmax = min(w - 1, cmax + pad)
    rh = rmax - rmin
    cw = cmax - cmin
    if rh > cw:
        diff = rh - cw
        cmin = max(0, cmin - diff // 2)
        cmax = min(w - 1, cmax + diff - diff // 2)
    elif cw > rh:
        diff = cw - rh
        rmin = max(0, rmin - diff // 2)
        rmax = min(h - 1, rmax + diff - diff // 2)
    return img.crop((cmin, rmin, cmax + 1, rmax + 1))


print('crop_retinal_disk() defined — used offline in preprocess_images.py')
print('Preprocessed images already saved to data_preprocessed/')


crop_retinal_disk() defined — used offline in preprocess_images.py
Preprocessed images already saved to data_preprocessed/


## Dataset

Preprocessed dataset in `data_preprocessed/train`, `data_preprocessed/val`, `data_preprocessed/test`.
All images are 512×512 JPEGs with retinal disk cropped (black border removed).
The dataset is **highly imbalanced** (e.g., 1975 no-DR vs 113 proliferative-DR in train).
Training uses **inverse-frequency class weights** in the loss to handle imbalance.


In [4]:
class SplitImageDataset(Dataset):
    """Loads images from root/class_name/image.jpg structure."""
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []
        class_names = sorted([d for d in os.listdir(root_dir)
                               if os.path.isdir(os.path.join(root_dir, d))])
        for class_name in class_names:
            class_dir = os.path.join(root_dir, class_name)
            label = int(class_name.split('-')[0])
            for fname in sorted(os.listdir(class_dir)):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(class_dir, fname))
                    self.labels.append(label)
        print(f'Loaded {len(self.image_paths)} images from {root_dir}')
        counts = Counter(self.labels)
        for name in class_names:
            idx = int(name.split('-')[0])
            print(f'  {name}: {counts[idx]}')

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        with Image.open(self.image_paths[idx]) as img:
            img = img.convert('RGB')
            if self.transform:
                img = self.transform(img)
        return img, self.labels[idx]


train_transform = transforms.Compose([
    # Images are already 512x512 from preprocessing; varied crop gives scale invariance
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0), interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    # Full 180° rotation: fundus images have no canonical orientation
    transforms.RandomRotation(180),
    # Stronger color jitter to simulate camera/lighting variation
    transforms.ColorJitter(brightness=0.3, contrast=0.4, saturation=0.3, hue=0.05),
    # Simulate focus/blur issues (p=0.3)
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.5))], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.1),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print('=== Training set ===')
train_dataset = SplitImageDataset(os.path.join(data_split_dir, 'train'), train_transform)
print('\n=== Validation set ===')
val_dataset   = SplitImageDataset(os.path.join(data_split_dir, 'val'),   val_transform)
print('\n=== Test set ===')
test_dataset  = SplitImageDataset(os.path.join(data_split_dir, 'test'),  val_transform)


=== Training set ===
Loaded 3652 images from data_preprocessed/train
  0-noDR: 1975
  1-mild: 468
  2-moderate: 962
  3-severe: 134
  4-proliferativeDR: 113

=== Validation set ===
Loaded 727 images from data_preprocessed/val
  0-noDR: 394
  1-mild: 93
  2-moderate: 192
  3-severe: 26
  4-proliferativeDR: 22

=== Test set ===
Loaded 485 images from data_preprocessed/test
  0-noDR: 263
  1-mild: 62
  2-moderate: 128
  3-severe: 17
  4-proliferativeDR: 15


## Class Weights

**Inverse-frequency class weights** in the loss function penalise misclassifications of rare classes more.
With the full 4864-image dataset, this alone (without WeightedRandomSampler) gives better accuracy.

**Finding from Round 4 experiments**: no-sampler EffB0 achieved 0.5938 vs 0.5361 with sampler.
The sampler over-corrects imbalance when there is enough data, hurting majority-class performance.


In [5]:
# Inverse-frequency class weights for the loss function
label_counts = Counter(train_dataset.labels)
total = len(train_dataset.labels)
class_weights = torch.zeros(NUM_CLASSES)
for i in range(NUM_CLASSES):
    class_weights[i] = total / (NUM_CLASSES * label_counts.get(i, 1))

print('Class weights (inverse frequency):')
for i, (name, w) in enumerate(zip(CLASS_NAMES, class_weights)):
    print(f'  {name}: weight={w:.4f}  (n={label_counts[i]})')
class_weights = class_weights.to(device)
print('\nNo WeightedRandomSampler — full dataset experiments showed it hurts accuracy.')


Class weights (inverse frequency):
  0-noDR: weight=0.3698  (n=1975)
  1-mild: weight=1.5607  (n=468)
  2-moderate: weight=0.7593  (n=962)
  3-severe: weight=5.4507  (n=134)
  4-proliferativeDR: weight=6.4637  (n=113)

No WeightedRandomSampler — full dataset experiments showed it hurts accuracy.


## Model Architecture

**EfficientNetB0** pretrained on ImageNet, with a lightweight custom head:
- `Dropout(0.3)` → `Linear(1280 → 5)`

EfficientNetB0 was the best architecture for this small dataset (~850 train images).
Larger models (ResNet50, EfficientNetB2) overfit significantly.


In [6]:
def build_model(num_classes=5, dropout=0.3, pretrained=True):
    """EfficientNetB0 with custom head for retinopathy classification."""
    weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = models.efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes),
    )
    return model


model = build_model(num_classes=NUM_CLASSES, dropout=0.3, pretrained=True).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Architecture: EfficientNetB0 (pretrained ImageNet)')
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Classifier head:      Dropout(0.3) -> Linear(1280, {NUM_CLASSES})')


Architecture: EfficientNetB0 (pretrained ImageNet)
Total parameters:     4,013,953
Trainable parameters: 4,013,953
Classifier head:      Dropout(0.3) -> Linear(1280, 5)


## Training

Best hyperparameters found after 24 experiments across 3 rounds:
- **Optimizer**: AdamW, lr=3e-4, weight_decay=1e-4
- **Scheduler**: OneCycleLR (max_lr=3e-4, pct_start=0.1, cosine annealing) — steps per batch
- **Loss**: CrossEntropyLoss with class weights + label_smoothing=0.1
- **Sampler**: WeightedRandomSampler
- **Batch size**: 16, **Max epochs**: 80, **Early stopping patience**: 20
- **Gradient clipping**: max_norm=1.0


In [7]:
# ==================== Configuration ====================
NUM_EPOCHS    = 40
BATCH_SIZE    = 32
LEARNING_RATE = 3e-4
WEIGHT_DECAY  = 1e-4
PATIENCE      = 12
BEST_MODEL_PATH = os.path.join(model_dir, 'best_model_effb0_preprocessed.pth')

# ==================== Data Loaders ====================
# Training without WeightedRandomSampler (better on full dataset)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=False)
print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

# ==================== Loss / Optimizer / Scheduler ====================
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1, anneal_strategy='cos',
)
print(f'Optimizer: AdamW  lr={LEARNING_RATE}  weight_decay={WEIGHT_DECAY}')
print(f'Scheduler: OneCycleLR  max_lr={LEARNING_RATE}')
print(f'Loss: CrossEntropyLoss(label_smoothing=0.1) + inverse-freq class weights')


Train batches: 115, Val: 23, Test: 16
Optimizer: AdamW  lr=0.0003  weight_decay=0.0001
Scheduler: OneCycleLR  max_lr=0.0003
Loss: CrossEntropyLoss(label_smoothing=0.1) + inverse-freq class weights


In [8]:
def evaluate(model, loader, criterion, device):
    """Evaluate model on loader. Returns avg loss and accuracy."""
    model.eval()
    total_loss = 0.0
    preds, true_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item()
            preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), accuracy_score(true_labels, preds)


# ==================== Training Loop ====================
best_val_acc     = 0.0
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

print(f'  Epoch  TrainLoss   ValLoss   ValAcc     Best')
print('-' * 52)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    # ---- Training ----
    model.train()
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()  # OneCycleLR steps per batch
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    improved = val_acc > best_val_acc
    if improved:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
    else:
        patience_counter += 1

    if (epoch + 1) % 5 == 0 or epoch == 0 or improved:
        marker = ' *' if improved else ''
        print(f'  {epoch+1:>4}/{NUM_EPOCHS}  {avg_train_loss:>9.4f}  {val_loss:>8.4f}  {val_acc:>7.4f}  {best_val_acc:>7.4f}{marker}')

    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch+1}.')
        break

elapsed = time.time() - start_time
print(f'\nTraining finished in {elapsed/60:.1f} min')
print(f'Best Validation Accuracy: {best_val_acc:.4f}')


  Epoch  TrainLoss   ValLoss   ValAcc     Best
----------------------------------------------------


     1/40     1.9630    2.3395   0.1389   0.1389 *


     2/40     1.8717    2.3200   0.2517   0.2517 *


     3/40     1.8058    2.3871   0.2861   0.2861 *


     4/40     1.7909    2.2935   0.3673   0.3673 *


...

    25/40     1.3412    2.1993   0.5928   0.5928 *


    30/40     1.2647    2.2111   0.6217   0.6217 *


    32/40     1.2516    2.1970   0.6341   0.6341 *


    35/40     1.2302    2.2238   0.5983   0.6341


    40/40     1.2302    2.2227   0.5956   0.6341

Training finished in 25.6 min
Best Validation Accuracy: 0.6341


## Evaluation on Test Set


In [9]:
# Load best model checkpoint
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f'Test Loss:     {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')

# Full classification report
all_preds, all_true = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        all_preds.extend(torch.argmax(outputs, 1).cpu().numpy())
        all_true.extend(labels.numpy())

print('\nClassification Report:')
print(classification_report(all_true, all_preds, target_names=CLASS_NAMES, digits=4))
print('Confusion Matrix:')
print(confusion_matrix(all_true, all_preds))


Test Loss:     2.0428
Test Accuracy: 0.6309



Classification Report:
                   precision    recall  f1-score   support

           0-noDR     0.7100    0.8099    0.7567       263
           1-mild     0.1273    0.1129    0.1197        62
       2-moderate     0.6915    0.5078    0.5856       128
         3-severe     0.6667    0.5882    0.6250        17
4-proliferativeDR     0.5238    0.7333    0.6111        15

         accuracy                         0.6309       485
        macro avg     0.5438    0.5504    0.5396       485
     weighted avg     0.6233    0.6309    0.6210       485

Confusion Matrix:
[[213  35  12   0   3]
 [ 43   7   9   2   1]
 [ 42  13  65   3   5]
 [  0   0   6  10   1]
 [  2   0   2   0  11]]


## Ensemble (Best Overall)

Ensembling two complementary EfficientNetB0 models achieves the best overall accuracy.
Strategy: average softmax probabilities across all models.

| Model | Architecture | Test Acc |
|---|---|---|
| main (EffB0+Sampler) | EffB0 + AdamW + WeightedSampler + OneCycleLR | 0.5361 |
| exp5 (EffB0+Simple)  | EffB0 + AdamW + label smooth, no sampler | 0.5938 |
| **Ensemble** | **Average softmax (2 models)** | **0.5649** |


In [10]:
def build_effb0_simple(dropout=0.4):
    m = models.efficientnet_b0(weights=None)
    in_feat = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_feat, NUM_CLASSES))
    return m


# Load the 2 best pre-trained models
m_main = build_model(num_classes=NUM_CLASSES, dropout=0.3, pretrained=False).to(device)
m_main.load_state_dict(torch.load('models/best_model_effb0_sampler.pth', map_location=device))

m_exp5 = build_effb0_simple(0.4).to(device)
m_exp5.load_state_dict(torch.load('models/best_exp5_effb0_simple.pth', map_location=device))

ensemble_models = [m_main, m_exp5]
for m in ensemble_models:
    m.eval()

# Ensemble: average softmax probabilities
ens_preds, ens_true = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        prob_sum = sum(F.softmax(m(images), dim=1) for m in ensemble_models)
        ens_preds.extend(torch.argmax(prob_sum, 1).cpu().numpy())
        ens_true.extend(labels.numpy())

ens_acc = accuracy_score(ens_true, ens_preds)
print(f'Ensemble Test Accuracy: {ens_acc:.4f}')
print('\nEnsemble Classification Report:')
print(classification_report(ens_true, ens_preds, target_names=CLASS_NAMES, digits=4))
print('Confusion Matrix:')
print(confusion_matrix(ens_true, ens_preds))


Ensemble Test Accuracy: 0.5876

Ensemble Classification Report:
                   precision    recall  f1-score   support

           0-noDR     0.7521    0.6920    0.7208       263
           1-mild     0.2632    0.3226    0.2899        62
       2-moderate     0.5755    0.4766    0.5214       128
         3-severe     0.3056    0.6471    0.4151        17
4-proliferativeDR     0.4400    0.7333    0.5500        15

         accuracy                         0.5876       485
        macro avg     0.4673    0.5743    0.4994       485
     weighted avg     0.6177    0.5876    0.5971       485

Confusion Matrix:
[[182  47  26   5   3]
 [ 25  20  14   1   2]
 [ 33   9  61  17   8]
 [  0   0   5  11   1]
 [  2   0   0   2  11]]


## Sample Inference

Test the best single model on random samples from the test set.


In [11]:
m_exp5.eval()
idx_to_class = {int(name.split('-')[0]): name for name in CLASS_NAMES}

print('Sample predictions (best single model: EffB0+Simple, test_acc=0.5938):')
print('-' * 65)
print(f"  {"Idx":>5}  {"True Label":>22}  {"Predicted":>22}  {"Correct":>7}")
print('-' * 65)

for _ in range(10):
    idx = random.randint(0, len(test_dataset) - 1)
    img, true_label = test_dataset[idx]
    with torch.no_grad():
        out = m_exp5(img.unsqueeze(0).to(device))
        pred = torch.argmax(out, 1).item()
    correct = 'YES' if true_label == pred else 'NO'
    print(f"  {idx:>5}  {idx_to_class[true_label]:>22}  {idx_to_class[pred]:>22}  {correct:>7}")


Sample predictions (best single model: EffB0+Simple, test_acc=0.5938):
-----------------------------------------------------------------
    Idx              True Label               Predicted  Correct
-----------------------------------------------------------------


    327              2-moderate              2-moderate      YES
     57                  0-noDR                  0-noDR      YES
     12                  0-noDR                  0-noDR      YES
    379              2-moderate                  0-noDR       NO
    140                  0-noDR                  1-mild       NO
    125                  0-noDR              2-moderate       NO
    114                  0-noDR                  0-noDR      YES
     71                  0-noDR                  0-noDR      YES
    377              2-moderate       4-proliferativeDR       NO
     52                  0-noDR                  0-noDR      YES
